<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/CSC791_DLBA_VGG_FGSM_Transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer-based black-box adversarial attack between the dense and pruned models

In [1]:
import tensorflow as tf
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import torch
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import torchvision

import copy
import math
from typing import List, Tuple
import time


import cv2
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def get_train_transform(transform_size=224):
  train_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return train_transform

def get_test_transform(transform_size):
  test_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return test_transform


In [5]:
def get_train_val_split(dataset='10', val_ratio=0.1, seed=42):
  if dataset == '10':
    full_train_for_indices = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
  else:
    full_train_for_indices = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=None)

  num_train = len(full_train_for_indices)
  num_val = int(val_ratio * num_train)
  num_train_split = num_train - num_val

  generator = torch.Generator().manual_seed(seed)
  permuted_indices = torch.randperm(num_train, generator=generator).tolist()

  train_indices = permuted_indices[:num_train_split]
  val_indices = permuted_indices[num_train_split:]

  return train_indices, val_indices


In [6]:
# from torchvision.datasets.vision import data
def load_dataset(dataset='10', val_ratio=0.1, seed=42, batch_size=64):
  train_transform = get_train_transform(transform_size=224)
  test_transform = get_test_transform(transform_size=224)

  if dataset == '10':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

    print('CIFAR-10 dataset downloaded and loaded successfully.')

  elif dataset == '100':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR100(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('beaver', 'dolphin', 'otter', 'seal', 'whale', 'aquarium fish', 'flatfish', 'ray', 'shark', 'trout', 'orchids', 'poppies', 'roses', 'sunflowers', 'tulips', 'bottles', 'bowls', 'cans', 'cups', 'plates', 'apples', 'mushrooms', 'oranges', 'pears', 'sweet peppers', 'clock', 'computer keyboard', 'lamp', 'telephone', 'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach', 'bear', 'leopard', 'lion', 'tiger', 'wolf', 'bridge', 'castle', 'house', 'road', 'skyscraper', 'cloud', 'forest', 'mountain', 'plain', 'sea', 'camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk', 'crab', 'lobster', 'snail', 'spider', 'worm', 'baby', 'boy', 'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake', 'turtle', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'maple', 'oak', 'palm', 'pine', 'willow', 'bicycle', 'bus', 'motorcycle', 'pickup truck', 'train', 'lawn-mower', 'rocket', 'streetcar', 'tank', 'tractor')
    print('CIFAR-100 dataset downloaded and loaded successfully.')

  return trainset, trainloader, testset, testloader, valset, valloader, classes


In [7]:
def customize_model(model, num_classes):
  model.classifier[6] = nn.Linear(4096, num_classes)
  return model


In [8]:
def get_model(num_classes, path=''):
  if path != '':
    model = models.vgg16(weights=None)
    model = customize_model(model, num_classes)
    print(f'Loading model from path {path}')
    model.load_state_dict(torch.load(path, map_location=device))
  else:
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    model = customize_model(model, num_classes)
  model = model.to(device)
  return model

In [9]:
def get_normalized_bounds(mean, std, device):
    """
    What this does:
    - Converts the valid raw pixel range [0, 1] into normalized-space bounds
    - These bounds are used to clamp adversarial images correctly
    """
    mean = mean.to(device).view(1, 3, 1, 1)
    std = std.to(device).view(1, 3, 1, 1)

    clamp_min = (0.0 - mean) / std
    clamp_max = (1.0 - mean) / std
    return clamp_min, clamp_max


def fgsm_attack_normalized(images, epsilon, data_grad, std):
    """
    What this does:
    - Applies FGSM to normalized images
    - Interprets epsilon in raw pixel space, then rescales it channel-wise
    - Returns adversarial images in normalized space
    """
    std = std.to(images.device).view(1, 3, 1, 1)
    epsilon_normalized = epsilon / std
    adv_images = images + epsilon_normalized * data_grad.sign()
    return adv_images



def init_transfer_stats():
    return {
        "total": 0,
        "source_clean_correct": 0,
        "target_clean_correct": 0,
        "source_adv_correct": 0,
        "target_adv_correct": 0,
        "target_initially_correct": 0,
        "target_transfer_success": 0,
    }


def finalize_transfer_stats(stats):
    total = stats["total"]
    return {
        "source_clean_acc": stats["source_clean_correct"] / total,
        "target_clean_acc": stats["target_clean_correct"] / total,
        "source_adv_acc": stats["source_adv_correct"] / total,
        "target_adv_acc": stats["target_adv_correct"] / total,
        "target_transfer_fooling_rate": (
            stats["target_transfer_success"] / stats["target_initially_correct"]
            if stats["target_initially_correct"] > 0 else 0.0
        ),
    }


def update_transfer_stats(
    stats,
    labels,
    source_clean_preds,
    target_clean_preds,
    source_adv_preds,
    target_adv_preds
):
    batch_size = labels.size(0)
    stats["total"] += batch_size

    stats["source_clean_correct"] += (source_clean_preds == labels).sum().item()
    stats["target_clean_correct"] += (target_clean_preds == labels).sum().item()
    stats["source_adv_correct"] += (source_adv_preds == labels).sum().item()
    stats["target_adv_correct"] += (target_adv_preds == labels).sum().item()

    target_correct_mask = (target_clean_preds == labels)
    target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

    stats["target_initially_correct"] += target_correct_mask.sum().item()
    stats["target_transfer_success"] += target_transfer_success_mask.sum().item()

In [10]:
def evaluate_dense_to_many_pruned_streaming(
    dense_model,
    pruned_models_dict,
    testloader,
    epsilon,
    device,
    mean,
    std
):
    """
    Generates dense-crafted FGSM once per batch, then evaluates that same batch
    on all pruned target models before moving on.

    pruned_models_dict:
        {
            sparsity1: pruned_model1,
            sparsity2: pruned_model2,
            ...
        }
    """
    dense_model.eval()
    for model in pruned_models_dict.values():
        model.eval()

    criterion = nn.CrossEntropyLoss()
    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    stats_dict = {sparsity: init_transfer_stats() for sparsity in pruned_models_dict}

    for batch_idx, (images, labels) in enumerate(testloader):
        if batch_idx % 20 == 0:
          print(f"[Dense->Many] Batch {batch_idx}")

        images = images.to(device)
        labels = labels.to(device)

        # Dense clean + gradient
        images_for_grad = images.detach().clone().requires_grad_(True)
        dense_model.zero_grad(set_to_none=True)

        dense_clean_outputs = dense_model(images_for_grad)
        dense_clean_preds = dense_clean_outputs.argmax(dim=1)

        loss = criterion(dense_clean_outputs, labels)
        loss.backward()

        data_grad = images_for_grad.grad.detach()
        adv_images = fgsm_attack_normalized(images_for_grad, epsilon, data_grad, std)
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min).detach()

        with torch.no_grad():
            dense_adv_outputs = dense_model(adv_images)
            dense_adv_preds = dense_adv_outputs.argmax(dim=1)

            for sparsity, pruned_model in pruned_models_dict.items():
                pruned_clean_outputs = pruned_model(images)
                pruned_adv_outputs = pruned_model(adv_images)

                pruned_clean_preds = pruned_clean_outputs.argmax(dim=1)
                pruned_adv_preds = pruned_adv_outputs.argmax(dim=1)

                update_transfer_stats(
                    stats_dict[sparsity],
                    labels,
                    dense_clean_preds,
                    pruned_clean_preds,
                    dense_adv_preds,
                    pruned_adv_preds
                )

        del images, labels, images_for_grad
        del dense_clean_outputs, dense_adv_outputs, dense_clean_preds, dense_adv_preds
        del loss, data_grad, adv_images

    return {s: finalize_transfer_stats(stats) for s, stats in stats_dict.items()}

In [11]:
def evaluate_transfer_attack(
    source_model,
    target_model,
    testloader,
    epsilon,
    device,
    mean,
    std,
    save_per_batch=0
):
    """
    Generates FGSM examples on source_model and immediately evaluates on target_model.

    Use this when source_model is changing and you do not want caching.
    """
    source_model.eval()
    target_model.eval()
    criterion = nn.CrossEntropyLoss()

    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    total = 0
    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct = 0
    target_adv_correct = 0

    target_initially_correct = 0
    target_transfer_success = 0

    saved_examples = []

    for batch_idx, (images, labels) in enumerate(testloader):
        if batch_idx % 20 == 0:
          print(f"[Direct transfer] Batch {batch_idx}")

        images = images.to(device)
        labels = labels.to(device)

        # Clean target predictions do NOT need gradients
        with torch.no_grad():
            target_clean_outputs = target_model(images)
            target_clean_preds = target_clean_outputs.argmax(dim=1)

        # Source branch needs gradients
        images_for_grad = images.detach().clone().requires_grad_(True)

        source_model.zero_grad(set_to_none=True)

        source_clean_outputs = source_model(images_for_grad)
        source_clean_preds = source_clean_outputs.argmax(dim=1)

        loss = criterion(source_clean_outputs, labels)
        loss.backward()

        data_grad = images_for_grad.grad.detach()

        adv_images = fgsm_attack_normalized(images_for_grad, epsilon, data_grad, std)
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min).detach()

        with torch.no_grad():
            source_adv_outputs = source_model(adv_images)
            target_adv_outputs = target_model(adv_images)

            source_adv_preds = source_adv_outputs.argmax(dim=1)
            target_adv_preds = target_adv_outputs.argmax(dim=1)

        batch_size = labels.size(0)
        total += batch_size

        source_clean_correct += (source_clean_preds == labels).sum().item()
        target_clean_correct += (target_clean_preds == labels).sum().item()
        source_adv_correct += (source_adv_preds == labels).sum().item()
        target_adv_correct += (target_adv_preds == labels).sum().item()

        target_correct_mask = (target_clean_preds == labels)
        target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

        target_initially_correct += target_correct_mask.sum().item()
        target_transfer_success += target_transfer_success_mask.sum().item()

        if save_per_batch > 0:
            idxs = target_transfer_success_mask.nonzero(as_tuple=True)[0][:save_per_batch]
            for idx in idxs:
                saved_examples.append({
                    "clean_image": images[idx].detach().cpu(),
                    "adv_image": adv_images[idx].detach().cpu(),
                    "label": labels[idx].item(),
                    "source_clean_pred": source_clean_preds[idx].item(),
                    "source_adv_pred": source_adv_preds[idx].item(),
                    "target_clean_pred": target_clean_preds[idx].item(),
                    "target_adv_pred": target_adv_preds[idx].item(),
                })

        del images, labels, images_for_grad
        del source_clean_outputs, target_clean_outputs
        del source_adv_outputs, target_adv_outputs
        del source_clean_preds, target_clean_preds
        del source_adv_preds, target_adv_preds
        del loss, data_grad, adv_images

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc = source_adv_correct / total
    target_adv_acc = target_adv_correct / total

    target_transfer_fooling_rate = (
        target_transfer_success / target_initially_correct
        if target_initially_correct > 0 else 0.0
    )

    return {
        "source_clean_acc": source_clean_acc,
        "target_clean_acc": target_clean_acc,
        "source_adv_acc": source_adv_acc,
        "target_adv_acc": target_adv_acc,
        "target_transfer_fooling_rate": target_transfer_fooling_rate,
        "saved_examples": saved_examples,
    }


In [12]:
# Same normalization as your CIFAR test loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])

In [13]:
num_classes = 100
dataset = '100'

In [14]:
load_path = f'/content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar{dataset}_two_stage_finetune.pth' # for fine tuned model only
dense_model = get_model(num_classes=num_classes, path=load_path)
trainset, trainloader, testset, testloader, valset, valloader, classes = load_dataset(dataset = dataset)
pruned_load_path_string = f'/content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar{dataset}_prune_finetune_'
model_sparsities = [20,40,60,80]
epsilons = [4/255, 8/255, 16/255]
# model_sparsities = [20]
# epsilons = [2/255]

Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_two_stage_finetune.pth


100%|██████████| 169M/169M [00:04<00:00, 35.2MB/s]


CIFAR-100 dataset downloaded and loaded successfully.


In [ ]:
for epsilon in epsilons:
    print(f"\n================ Epsilon: {epsilon} ================\n")

    # Load all pruned models once
    pruned_models = {}
    for sparsity in model_sparsities:
        pruned_load_path = pruned_load_path_string + str(sparsity) + ".pth"
        model = get_model(num_classes=num_classes, path=pruned_load_path)
        model = model.to(device)
        model.eval()
        pruned_models[sparsity] = model

    # Dense -> all pruned in one pass over testloader
    dense_to_all_results = evaluate_dense_to_many_pruned_streaming(
        dense_model=dense_model,
        pruned_models_dict=pruned_models,
        testloader=testloader,
        epsilon=epsilon,
        device=device,
        mean=CIFAR_MEAN,
        std=CIFAR_STD
    )

    for sparsity in model_sparsities:
        print(f"\nDense -> Pruned | Sparsity: {sparsity}, Epsilon: {epsilon}")
        res = dense_to_all_results[sparsity]
        print(f"Source clean acc: {res['source_clean_acc']:.4f}")
        print(f"Target clean acc: {res['target_clean_acc']:.4f}")
        print(f"Source adv acc:   {res['source_adv_acc']:.4f}")
        print(f"Target adv acc:   {res['target_adv_acc']:.4f}")
        print(f"Transfer fooling rate: {res['target_transfer_fooling_rate']:.4f}")

    # Pruned -> Dense still one source at a time
    for sparsity in model_sparsities:
        print(f"\nPruned -> Dense | Sparsity: {sparsity}, Epsilon: {epsilon}")

        pruned_to_dense = evaluate_transfer_attack(
            source_model=pruned_models[sparsity],
            target_model=dense_model,
            testloader=testloader,
            epsilon=epsilon,
            device=device,
            save_per_batch=0,
            mean=CIFAR_MEAN,
            std=CIFAR_STD
        )

        print(f"Source clean acc: {pruned_to_dense['source_clean_acc']:.4f}")
        print(f"Target clean acc: {pruned_to_dense['target_clean_acc']:.4f}")
        print(f"Source adv acc:   {pruned_to_dense['source_adv_acc']:.4f}")
        print(f"Target adv acc:   {pruned_to_dense['target_adv_acc']:.4f}")
        print(f"Transfer fooling rate: {pruned_to_dense['target_transfer_fooling_rate']:.4f}")

    # Cleanup
    for model in pruned_models.values():
        del model
    del pruned_models
    torch.cuda.empty_cache()


================ Epsilon: 0.01568627450980392 ================

Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_prune_finetune_20.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_prune_finetune_40.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_prune_finetune_60.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/vgg16_cifar100_prune_finetune_80.pth
[Dense->Many] Batch 0
[Dense->Many] Batch 20
[Dense->Many] Batch 40
[Dense->Many] Batch 60
[Dense->Many] Batch 80
[Dense->Many] Batch 100
[Dense->Many] Batch 120
[Dense->Many] Batch 140

Dense -> Pruned | Sparsity: 20, Epsilon: 0.01568627450980392
Source clean acc: 0.7263
Target clean acc: 0.7284
Source adv acc:   0.0679
Target adv acc:   0.0660
Transfer fooling rate: 0.9117

Dense -> Pruned | Sparsity: 40, Epsilon: 0.01568627450980392
Source clean acc: 0.7263
Target clean acc: 0.7337
Source adv acc:   0.0679
Targ